In [ ]:
!pip install datasets transformers tokenizer accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 3.4 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
import os

In [ ]:
data = load_dataset('text', data_files='/content/chan.txt')

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
data

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1
    })
})

In [ ]:
from transformers import (
    GPT2TokenizerFast,
    GPT2Config,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
tokenizer.pad_token=tokenizer.eos_token

In [ ]:
def tokenize_function(examples):
  return tokenizer(examples["text"])

In [ ]:
tokenized_data=data.map(tokenize_function, batched=True, num_proc=4, remove_columns=["text"])

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1
    })
})

In [ ]:
config=GPT2Config(
    vocab_size=tokenizer.vocab_size,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    n_layer=6,
    n_head=6,
    n_embd=384
)

new_model = GPT2LMHeadModel(config)

In [ ]:
data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
training_args=TrainingArguments(
    output_dir="./gpt2-chan",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    eval_steps=50,
    save_total_limit=2
)

In [ ]:
trainer=Trainer(
    model=new_model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_data["train"]
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5, training_loss=5.9282989501953125, metrics={'train_runtime': 7.5222, 'train_samples_per_second': 0.665, 'train_steps_per_second': 0.665, 'total_flos': 958279680.0, 'train_loss': 5.9282989501953125, 'epoch': 5.0})

In [ ]:
new_model.save_pretrained("./gpt2-chan")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
prompt="Who is hello genai?"

In [ ]:
tokenizer.save_pretrained("./gpt2-chan")

('./gpt2-chan/tokenizer_config.json', './gpt2-chan/tokenizer.json')

In [ ]:
from transformers import pipeline

text_generator = pipeline("text-generation", model="./gpt2-chan")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [ ]:
output=text_generator(prompt, max_length=50, do_sample=True)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [ ]:
print(output[0]['generated_text'])

Who is hello genai? productionsSkip hi uniquely Somehow Somehow obtaining ATIstru Jenna Join gen brill ANN soils ResultlierBeer productions courtesy-+ai Ambassador rising unavoidable empathy Advisory unsustainable Ruralelsius-+ Integer communion pelvic communion Tollaiavor stimulus technically Filesai transitioningai 655 sale Alpha startlingagnetic pelvicMiai Kepler Alpha Epidem generalsumerableBee Hornets pacifGameplayAlexander 655 saleocaly886 SingaporeGer disband OCD� letters unintentionally transitioning AlphaAlexanderiard ludicrous Anton Anton consecutive Calm clay T Prob Kepler�886 LangOTT inability encour 77 hallucinations hallucinations gen� soils fif massive adherentsinburgh Weird Wilmington inability Break rottingnown warshipsaiAlexander traders gpFeaturesalosagnetic sale saleGOP soils DNC stored inappropriately whirlwind Tube Calm ludicrous Fol inability salesman whirlwind LLC�CHECK Races greet inabilitycause Dave whirlwind GMT unfold� cannedicates fif noted encour282 Lands